# Step 1b - Audit of candidate payload corpora

Click **Runtime > Run all**. `audit_payload_candidates.md` downloads when it finishes. Send that file back.

Some of these may be gated or missing; errors are written into the report, so send it either way.

In [ ]:
!pip install -q datasets huggingface_hub pandas

Only if a dataset turns out to be gated: uncomment the two lines below, run the cell, and paste your Hugging Face token.

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()

In [ ]:
import json
import traceback

import pandas as pd
from datasets import (get_dataset_config_names, get_dataset_split_names,
                      load_dataset, load_dataset_builder)
from huggingface_hub import HfApi

SOURCES = [
    ("payloads", "deepset/prompt-injections"),
    ("payloads", "Lakera/gandalf_ignore_instructions"),
    ("payloads", "microsoft/llmail-inject-challenge"),
    ("payloads", "hackaprompt/hackaprompt-dataset"),
]
SAMPLE_ROWS = 3000      # rows read per split (streamed, so nothing large is downloaded)
MAX_CONFIGS = 6
OUT = "audit_payload_candidates.md"

lines = []
def w(s=""):
    print(s)
    lines.append(s)

def clip(v, n=220):
    s = v if isinstance(v, str) else json.dumps(v, ensure_ascii=False, default=str)
    s = s.replace("\n", " \\n ").replace("|", "\\|")
    return s if len(s) <= n else s[:n] + " ...[cut]"

def profile(rows):
    """Describe a list of dict rows: columns, text lengths, label-like value counts, samples."""
    df = pd.DataFrame(rows)
    w(f"Rows sampled: {len(df)}")
    w()
    w("| Column | Type | Missing | Unique | Mean chars | Max chars |")
    w("|---|---|---|---|---|---|")
    label_like = []
    for c in df.columns:
        col = df[c]
        as_str = col.map(lambda v: v if isinstance(v, str) else json.dumps(v, default=str))
        nunique = as_str.nunique()
        is_text = col.map(lambda v: isinstance(v, str)).mean() > 0.9
        missing = int(col.isna().sum() + (as_str.str.strip() == "").sum())
        mean_len = f"{as_str.str.len().mean():.0f}" if is_text else ""
        max_len = f"{as_str.str.len().max()}" if is_text else ""
        w(f"| `{c}` | {type(col.iloc[0]).__name__} | {missing} | {nunique} | {mean_len} | {max_len} |")
        if nunique <= 25 and as_str.str.len().mean() < 40:
            label_like.append((c, as_str))
    for c, as_str in label_like:
        w()
        w(f"Values of `{c}`:")
        w()
        w("| Value | Count |")
        w("|---|---|")
        for val, n in as_str.value_counts().items():
            w(f"| {clip(val, 80)} | {n} |")
    w()
    w("Sample rows:")
    for i, r in enumerate(rows[:3]):
        w()
        w(f"Row {i}:")
        for k, v in r.items():
            w(f"- `{k}`: {clip(v)}")
    # exact duplicates on the longest text column
    text_cols = [c for c in df.columns if df[c].map(lambda v: isinstance(v, str)).mean() > 0.9]
    if text_cols:
        main = max(text_cols, key=lambda c: df[c].str.len().mean())
        dup = int(df[main].duplicated().sum())
        w()
        w(f"Longest text column: `{main}`; exact duplicates in sample: {dup}")

def audit(role, repo):
    w(f"## {repo}")
    w()
    w(f"Role: {role}")
    try:
        info = HfApi().dataset_info(repo, files_metadata=True)
        card = info.card_data.to_dict() if info.card_data else {}
        w(f"License: {card.get('license', 'NOT STATED')}")
        w(f"Gated: {info.gated}")
        w(f"Last modified: {info.last_modified}")
        w()
        w("| File | Size (MB) |")
        w("|---|---|")
        for s in info.siblings[:40]:
            w(f"| {s.rfilename} | {(s.size or 0) / 1e6:.2f} |")
    except Exception as e:
        w(f"**Could not read repository info:** {e!r}")
    try:
        configs = get_dataset_config_names(repo)
    except Exception as e:
        w()
        w(f"**Could not list configs:** {e!r}")
        w("```")
        w(traceback.format_exc()[-1500:])
        w("```")
        return
    w()
    w(f"Configs: {configs}")
    for cfg in configs[:MAX_CONFIGS]:
        try:
            declared = {}
            try:
                sp = load_dataset_builder(repo, cfg).info.splits or {}
                declared = {k: v.num_examples for k, v in sp.items()}
            except Exception:
                pass
            for split in get_dataset_split_names(repo, cfg):
                w()
                w(f"### config `{cfg}`, split `{split}`")
                w()
                w(f"Declared row count: {declared.get(split, 'unknown')}")
                ds = load_dataset(repo, cfg, split=split, streaming=True)
                rows = []
                for r in ds:
                    rows.append(r)
                    if len(rows) >= SAMPLE_ROWS:
                        break
                profile(rows)
        except Exception as e:
            w()
            w(f"**Failed on config `{cfg}`:** {e!r}")
            w("```")
            w(traceback.format_exc()[-1500:])
            w("```")
    w()


In [ ]:
w("# Step 1b audit: candidate payload corpora")
w()
for role, repo in SOURCES:
    audit(role, repo)
with open(OUT, "w", encoding="utf-8") as f:
    f.write("\n".join(lines))
print(f"\nWrote {OUT}")

try:
    from google.colab import files
    files.download(OUT)
except ImportError:
    pass